# Classical ML Pipeline Clean

Notebook propre et complet pour la partie Machine Learning classique.

Pipeline applique a chaque image :
1. Pretraitement
2. Segmentation / contours
3. Extraction des features
4. Classification ML
5. Evaluation et comparaison


## 1. Imports


In [ ]:
from pathlib import Path
import sys
import json
import pickle
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


## 2. Project Setup


In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / 'src'))

from plant_disease.config import load_config
from plant_disease.dataset import build_index, stratified_split, save_metadata
from plant_disease.utils import ensure_dir, set_seed

config = load_config(ROOT / 'configs/default.yaml')
config.paths.dataset_root = (ROOT / 'data' / 'PlantVillage').resolve()
set_seed(config.seed)

print('Project root:', ROOT)
print('Dataset root:', config.paths.dataset_root)


## 3. Select The 4 Classes


In [ ]:
selected_classes = [
    'Tomato_healthy',
    'Tomato_Early_blight',
    'Tomato_Late_blight',
    'Tomato_Bacterial_spot',
]
config.dataset.classes = selected_classes
for class_name in selected_classes:
    print(class_name, (config.paths.dataset_root / class_name).exists())


## 4. Build Dataset Index And Splits


In [ ]:
rows = build_index(config)
split_rows = stratified_split(
    rows,
    train_ratio=config.dataset.train_ratio,
    val_ratio=config.dataset.val_ratio,
    seed=config.seed,
)
metadata_path = save_metadata(config, split_rows)
print('Metadata saved to:', metadata_path)
print('Total images:', len(split_rows))


## 5. Class Distribution


In [ ]:
class_counts = {label: sum(1 for row in rows if row['label'] == label) for label in selected_classes}
total = sum(class_counts.values())
summary_df = pd.DataFrame({
    'Class': list(class_counts.keys()),
    'Count': list(class_counts.values()),
})
summary_df['Percentage'] = (summary_df['Count'] / total * 100).round(2)
summary_df


In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.barplot(data=summary_df, y='Class', x='Count', palette='viridis')
for i, v in enumerate(summary_df['Count']):
    ax.text(v + 5, i, str(v), va='center')
plt.title('Class Distribution')
plt.xlabel('Count')
plt.ylabel('Class')
plt.tight_layout()
plt.show()


## 6. Sample Images Grid


In [ ]:
def load_rgb_image(path, image_size=224):
    image = cv2.imread(str(path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_AREA)

sample_paths = []
for label in selected_classes:
    class_paths = [Path(r['image_path']) for r in rows if r['label'] == label][:3]
    for p in class_paths:
        sample_paths.append((label, p))

fig, axes = plt.subplots(4, 3, figsize=(10, 12))
for ax, (label, path) in zip(axes.flatten(), sample_paths):
    ax.imshow(load_rgb_image(path, 224))
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 7. Helper Functions


In [ ]:
def to_gray(image):
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

def to_hsv(image):
    return cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

def to_lab(image):
    return cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

def preprocess_image_local(image):
    denoised = cv2.GaussianBlur(image, (3, 3), 0)
    lab = to_lab(denoised)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_l = clahe.apply(l_channel)
    merged = cv2.merge([enhanced_l, a_channel, b_channel])
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)


## 8. Preprocessing Example On 3 Images


In [ ]:
example_paths = [
    next(Path(r['image_path']) for r in rows if r['label'] == 'Tomato_healthy'),
    next(Path(r['image_path']) for r in rows if r['label'] == 'Tomato_Early_blight'),
    next(Path(r['image_path']) for r in rows if r['label'] == 'Tomato_Late_blight'),
]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, p in enumerate(example_paths):
    original = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    resized = load_rgb_image(p, 224)
    preprocessed = preprocess_image_local(resized)
    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f'Original {original.shape}')
    axes[i, 1].imshow(resized)
    axes[i, 1].set_title('Resized 224x224')
    axes[i, 2].imshow(preprocessed)
    axes[i, 2].set_title('Filtered + CLAHE')
    for j in range(3):
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()


## 9. Color Space Conversion


In [ ]:
img = load_rgb_image(example_paths[1], 224)
gray = to_gray(img)
hsv = to_hsv(img)
lab = to_lab(img)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(img); axes[0].set_title('Original')
axes[1].imshow(gray, cmap='gray'); axes[1].set_title('GRAY')
axes[2].imshow(hsv); axes[2].set_title('HSV')
axes[3].imshow(lab); axes[3].set_title('LAB')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


HSV is useful because it separates chromatic information from intensity. On diseased leaves, changes in hue and saturation are often easier to isolate in HSV than in raw RGB, where brightness variations can hide color anomalies.


## 10. Histogram Analysis


In [ ]:
healthy = load_rgb_image(example_paths[0], 224)
diseased = load_rgb_image(example_paths[2], 224)
healthy_hsv = to_hsv(healthy)
diseased_hsv = to_hsv(diseased)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for c, color in enumerate(['r', 'g', 'b']):
    h_hist = cv2.calcHist([healthy], [c], None, [32], [0, 256]).flatten()
    d_hist = cv2.calcHist([diseased], [c], None, [32], [0, 256]).flatten()
    axes[0, 0].plot(h_hist, color=color)
    axes[0, 1].plot(d_hist, color=color)
axes[0, 0].set_title('Healthy RGB histogram')
axes[0, 1].set_title('Diseased RGB histogram')
for c, label in enumerate(['H', 'S', 'V']):
    upper = 180 if c == 0 else 256
    h_hist = cv2.calcHist([healthy_hsv], [c], None, [32], [0, upper]).flatten()
    d_hist = cv2.calcHist([diseased_hsv], [c], None, [32], [0, upper]).flatten()
    axes[1, 0].plot(h_hist, label=label)
    axes[1, 1].plot(d_hist, label=label)
axes[1, 0].set_title('Healthy HSV histogram')
axes[1, 1].set_title('Diseased HSV histogram')
axes[1, 0].legend(); axes[1, 1].legend()
plt.tight_layout()
plt.show()


Observation: diseased leaves usually show histogram shifts caused by browning, yellowing, and loss of healthy green intensity. This is one reason histogram-based color features are useful for classical ML.


## 11. Segmentation And Contours Functions


In [ ]:
def threshold_mask(image):
    gray = to_gray(image)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask

def hsv_mask(image):
    hsv = to_hsv(image)
    lower_green = np.array([20, 20, 20], dtype=np.uint8)
    upper_green = np.array([110, 255, 255], dtype=np.uint8)
    return cv2.inRange(hsv, lower_green, upper_green)

def kmeans_mask(image, k=3):
    pixels = image.reshape((-1, 3)).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
    centers = centers.astype(np.uint8)
    green_scores = centers[:, 1].astype(np.int32) - (centers[:, 0].astype(np.int32) + centers[:, 2].astype(np.int32)) / 2
    leaf_cluster = int(np.argmax(green_scores))
    return (labels.flatten() == leaf_cluster).astype(np.uint8).reshape(image.shape[:2]) * 255

def sobel_edges(image):
    gray = to_gray(image)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    return cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def canny_edges(image):
    gray = to_gray(image)
    return cv2.Canny(gray, 80, 160)


## 12. Segmentation Comparison On Healthy And Diseased Images


In [ ]:
compare_imgs = [healthy, diseased]
compare_titles = ['Healthy', 'Diseased']
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for i, image in enumerate(compare_imgs):
    outputs = [image, threshold_mask(image), hsv_mask(image), kmeans_mask(image), canny_edges(image)]
    titles = [compare_titles[i], 'Otsu', 'HSV', 'KMeans', 'Canny']
    for j, (out, title) in enumerate(zip(outputs, titles)):
        axes[i, j].imshow(out, cmap='gray' if getattr(out, 'ndim', 3) == 2 else None)
        axes[i, j].set_title(title)
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()


Subjective comparison: HSV segmentation isolates leaf tissue more consistently than pure thresholding. KMeans is useful when the background is more complex. Canny gives sharper lesion contours, while Sobel responds more broadly to texture variation.


## 13. Feature Extraction Functions


In [ ]:
def glcm_features(gray, levels=16):
    scaled = (gray.astype(np.float32) / 256.0 * levels).astype(np.int32)
    scaled = np.clip(scaled, 0, levels - 1)
    glcm = np.zeros((levels, levels), dtype=np.float64)
    left = scaled[:, :-1]
    right = scaled[:, 1:]
    for i in range(levels):
        for j in range(levels):
            glcm[i, j] = np.sum((left == i) & (right == j))
    total = glcm.sum()
    if total == 0:
        return [0.0, 0.0, 0.0, 0.0]
    glcm /= total
    ii, jj = np.indices(glcm.shape)
    contrast = float(np.sum(glcm * (ii - jj) ** 2))
    homogeneity = float(np.sum(glcm / (1 + np.abs(ii - jj))))
    energy = float(np.sum(glcm ** 2))
    mean_i = np.sum(ii * glcm)
    mean_j = np.sum(jj * glcm)
    std_i = np.sqrt(np.sum(glcm * (ii - mean_i) ** 2))
    std_j = np.sqrt(np.sum(glcm * (jj - mean_j) ** 2))
    denom = float(std_i * std_j) if std_i > 0 and std_j > 0 else 1.0
    correlation = float(np.sum(glcm * (ii - mean_i) * (jj - mean_j)) / denom)
    return [contrast, homogeneity, energy, correlation]

def shape_features(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    largest = max(contours, key=cv2.contourArea) if contours else None
    if largest is None:
        return [0.0, 0.0, 0.0]
    area = float(cv2.contourArea(largest))
    perimeter = float(cv2.arcLength(largest, True))
    circularity = float((4.0 * np.pi * area) / (perimeter ** 2)) if perimeter > 0 else 0.0
    return [area, perimeter, circularity]

def extract_features(image, bins=16):
    preprocessed = preprocess_image_local(image)
    hsv = to_hsv(preprocessed)
    gray = to_gray(preprocessed)
    mask_hsv = hsv_mask(preprocessed)
    mask_otsu = threshold_mask(preprocessed)
    mask_kmeans = kmeans_mask(preprocessed)
    sobel = sobel_edges(preprocessed)
    canny = canny_edges(preprocessed)

    rgb_hist = []
    for c in range(3):
        hist = cv2.calcHist([preprocessed], [c], None, [bins], [0, 256]).flatten()
        rgb_hist.extend((hist / max(hist.sum(), 1.0)).tolist())

    hsv_hist = []
    ranges = [(0, 180), (0, 256), (0, 256)]
    for c in range(3):
        hist = cv2.calcHist([hsv], [c], None, [bins], list(ranges[c])).flatten()
        hsv_hist.extend((hist / max(hist.sum(), 1.0)).tolist())

    texture = glcm_features(gray)
    shape = shape_features(mask_hsv)
    extra = [
        float(np.count_nonzero(mask_otsu) / mask_otsu.size),
        float(np.count_nonzero(mask_hsv) / mask_hsv.size),
        float(np.count_nonzero(mask_kmeans) / mask_kmeans.size),
        float(np.count_nonzero(sobel) / sobel.size),
        float(np.count_nonzero(canny) / canny.size),
    ]
    return np.asarray(rgb_hist + hsv_hist + texture + shape + extra, dtype=np.float32)


## 14. Build Feature Matrices For Train, Val And Test


In [ ]:
label_to_idx = {label: idx for idx, label in enumerate(selected_classes)}

def build_feature_matrix(rows, split_name):
    X, y = [], []
    for row in rows:
        if row['split'] != split_name:
            continue
        image = load_rgb_image(Path(row['image_path']), config.dataset.image_size)
        features = extract_features(image, bins=16)
        X.append(features)
        y.append(label_to_idx[row['label']])
    return np.stack(X), np.asarray(y)

X_train, y_train = build_feature_matrix(split_rows, 'train')
X_val, y_val = build_feature_matrix(split_rows, 'val')
X_test, y_test = build_feature_matrix(split_rows, 'test')
print('X_train:', X_train.shape)
print('X_val:', X_val.shape)
print('X_test:', X_test.shape)


## 15. Candidate Models


In [ ]:
models = {
    'logistic_regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=3000, random_state=config.seed))]),
    'knn_5': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=5))]),
    'random_forest': RandomForestClassifier(n_estimators=300, random_state=config.seed, n_jobs=-1),
    'extra_trees': ExtraTreesClassifier(n_estimators=300, random_state=config.seed, n_jobs=-1),
    'svm_rbf': Pipeline([('scaler', StandardScaler()), ('model', SVC(C=3.0, kernel='rbf', gamma='scale', probability=True))]),
}


## 16. Validation Results


In [ ]:
validation_results = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    pred = model.predict(X_val)
    precision, recall, f1, _ = precision_recall_fscore_support(y_val, pred, average='macro')
    validation_results[name] = {
        'accuracy': accuracy_score(y_val, pred),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'train_time_sec': float(elapsed),
    }
validation_df = pd.DataFrame(validation_results).T.sort_values('f1', ascending=False)
validation_df


## 17. Final Training On Train + Val, Then Test


In [ ]:
best_model_name = validation_df.index[0]
best_model = models[best_model_name]

X_train_full = np.concatenate([X_train, X_val], axis=0)
y_train_full = np.concatenate([y_train, y_val], axis=0)

best_model.fit(X_train_full, y_train_full)
test_pred = best_model.predict(X_test)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, test_pred, average='macro')
test_accuracy = accuracy_score(y_test, test_pred)
print('Best model:', best_model_name)
print(classification_report(y_test, test_pred, target_names=selected_classes))


In [ ]:
cm = confusion_matrix(y_test, test_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=selected_classes).plot(ax=ax, cmap='Blues', xticks_rotation=45)
plt.tight_layout()
plt.show()


## 18. Save Results


In [ ]:
output_dir = ensure_dir(ROOT / 'artifacts' / 'classical_ml')
with open(output_dir / 'best_classical_model_clean.pkl', 'wb') as f:
    pickle.dump(best_model, f)

summary = {
    'selected_classes': selected_classes,
    'feature_dim': int(X_train.shape[1]),
    'validation_results': validation_results,
    'best_model_name': best_model_name,
    'test_accuracy': float(test_accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_macro_f1': float(f1),
    'test_confusion_matrix': cm.tolist(),
}
with open(output_dir / 'classical_clean_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print('Saved results in:', output_dir)
